In [51]:
import requests
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import statsmodels.api as sm
from datetime import datetime, timedelta
import json



## 1. Load data
### 1.1 SR data

In [3]:
def load_and_process_sr_data(start_date, end_date):
    # col names
    column_names = ['Date+T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10', 
                    'T11', 'T12', 'T13', 'T14', 'T15', 'F1', 'F2', 'F3', 'SR', 'A', 'Circ pump counter']

    dataframes = {}
    current_date = start_date

    while current_date <= end_date:
        file_name = f'data_{current_date.strftime("%Y-%m-%d")}.csv'
        date_key = current_date.strftime("%Y-%m-%d")

        try:
            # load the data 
            data = pd.read_csv(file_name, header=None, names=column_names)

            # handle 'Date+T1' if concatenated
            if '-' in data.iloc[0, 0]:
                new_cols = data['Date+T1'].str.split(' T1:', expand=True)
                data['Date'] = new_cols[0]
                data['T1'] = new_cols[1].astype(float)
            else:
                data = data.reset_index().rename(columns={'index': 'Date'})

            # Clean and select necessary columns
            data['SR'] = data['SR'].str.split(':').str[1].astype(float)
            data = data[['Date', 'SR']].dropna()

            # Store the DataFrame in the dictionary with the date as the key
            dataframes[date_key] = data
            
        except FileNotFoundError:
            print(f"File not found: {file_name}")
        except Exception as e:
            print(f"Error reading {file_name}: {e}")

        current_date += timedelta(days=1)

    # Concatenate all DataFrames
    recent_data = pd.concat(dataframes.values())
    recent_data['Date'] = pd.to_datetime(recent_data['Date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')
    recent_data['Dates'] = recent_data['Date'].dt.date
    recent_data['Time'] = recent_data['Date'].dt.time
    recent_data = recent_data[['Dates', 'Time', 'SR']].dropna().sort_values(by=['Dates'])
    return recent_data

# Example usage
start_date = datetime(2024, 5, 20)


In [4]:
current_date = datetime.now()
end_date = current_date - timedelta(days=1)
processed_data = load_and_process_sr_data(start_date, end_date)
processed_data

,Dates,Time,SR
0,2024-05-20,20:58:45,82.24
1240,2024-05-20,23:00:15,84.67
1239,2024-05-20,23:00:09,84.71
1238,2024-05-20,23:00:03,84.80
1237,2024-05-20,22:59:58,84.99
...,...,...,...
5256,2024-06-27,07:59:54,87.57
5257,2024-06-27,07:59:59,89.91
5258,2024-06-27,08:00:05,92.18
5260,2024-06-27,08:00:16,93.74


In [5]:
def process_backup_data(file_path):
    column_names = ['Date+T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10', 'T11', 'T12', 'T13', 'T14', 'T15', 'F1', 'F2', 'F3', 'SR', 'A', 'Circ pump counter']
    data = pd.read_csv(file_path, header=None, names=column_names, skiprows=1)
    data = data[['Date+T1', 'SR']]
    new_cols = data['Date+T1'].str.split(' T1:', expand=True)
    data['Date'] = new_cols[0]
    data.drop(columns=['Date+T1'], inplace=True)
    data['SR'] = data['SR'].str.split(':').str[1].astype(float)
    return data

backup0 = 'data_backup0.csv'
twentyfour_apr_may = process_backup_data(backup0)

# seperate the time from the date
twentyfour_apr_may['Date'] = pd.to_datetime(twentyfour_apr_may['Date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')
twentyfour_apr_may['Dates'] = twentyfour_apr_may['Date'].dt.date
twentyfour_apr_may['Time'] = twentyfour_apr_may['Date'].dt.time
twentyfour_apr_may = twentyfour_apr_may[['Dates', 'Time', 'SR']].dropna().sort_values(by=['Dates'])
twentyfour_apr_may 

,Dates,Time,SR
0,2024-04-20,04:01:34,80.57
8809,2024-04-20,17:19:34,218.51
8810,2024-04-20,17:19:39,225.78
8811,2024-04-20,17:19:45,218.27
8812,2024-04-20,17:19:50,240.25
...,...,...,...
415000,2024-05-17,06:12:48,82.13
415001,2024-05-17,06:12:54,88.20
415002,2024-05-17,06:12:59,90.38
414982,2024-05-17,06:11:02,83.52


In [7]:
all_SR = pd.concat([processed_data, twentyfour_apr_may], ignore_index=True).sort_values('Dates')
all_SR

,Dates,Time,SR
616621,2024-04-20,13:07:58,82.57
613949,2024-04-20,04:01:45,80.53
613950,2024-04-20,04:01:51,80.75
613951,2024-04-20,04:01:56,80.60
613952,2024-04-20,04:02:01,80.38
...,...,...,...
594642,2024-06-27,23:57:21,86.31
594643,2024-06-27,23:48:55,88.17
594644,2024-06-27,23:57:15,86.70
594646,2024-06-27,23:54:36,87.74


### 1.2 Weather Data

In [40]:
yesterday_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/yesterday?unitGroup=metric&key=HJ2JWKPKSPZKWD64FVT9F7A6Z&contentType=json"
last_7_day_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/last7days?unitGroup=metric&key=HJ2JWKPKSPZKWD64FVT9F7A6Z&contentType=json"
last_15_day_url =  "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/last15days?unitGroup=metric&key=HJ2JWKPKSPZKWD64FVT9F7A6Z&contentType=json"
tmr_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/tomorrow?unitGroup=metric&key=HJ2JWKPKSPZKWD64FVT9F7A6Z&contentType=json"
next_7_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/next7days?unitGroup=metric&key=HJ2JWKPKSPZKWD64FVT9F7A6Z&contentType=json"
next_24_hours_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/next24hours?unitGroup=metric&key=HJ2JWKPKSPZKWD64FVT9F7A6Z&contentType=json"

> Get tmr's weahter data to predict tmr's SR

In [34]:
# function to get tmr's weather
def get_tmr_weather_df():
    # URL setup for tomorrow's weather data
    tmr_date = datetime.now() + timedelta(days=1)
    tmr_response = requests.get(tmr_url)

    # Check if the request was successful
    if tmr_response.status_code == 200:
        # Parse the JSON data
        weather_data = tmr_response.json()
        print("Data retrieved successfully!")
        
        # File path setup for saving the data
        tmr_file_path = f"{tmr_date.strftime('%Y-%m-%d')}_weather_data.json"
        with open(tmr_file_path, 'w') as f:
            json.dump(weather_data, f, indent=4)
        print("Weather data saved successfully!")
        
        # Load and parse the JSON data into a DataFrame
        with open(tmr_file_path, 'r') as file:
            tmr_json_data = json.load(file)
        
        # Initialize an empty list to hold all hourly data
        tmr_hourly_data = []
        
        # Loop through each day in the data
        for day in tmr_json_data['days']:
            # Extract each 'hour' from 'hours' array and enrich it with 'day' level data
            for hour in day['hours']:
                hour.update({
                    'date': day['datetime'],
                    'tempmax': day['tempmax'],
                    'tempmin': day['tempmin'],
                    'feelslikemax': day['feelslikemax'],
                    'feelslikemin': day['feelslikemin'],
                    'sunrise': day['sunrise'],
                    'sunset': day['sunset'],
                    'moonphase': day['moonphase'],
                    'conditions': day['conditions'],
                    'description': day['description'],
                    'icon': day['icon']
                })
                tmr_hourly_data.append(hour)
        
        # Create a DataFrame from the hourly data
        tmr_hourly_df = pd.DataFrame(tmr_hourly_data)
        
        # Define the columns of interest
        columns_of_interest = [
            'date', 'datetime', 'temp', 'tempmax', 'tempmin', 'feelslike', 'feelslikemax', 'feelslikemin', 
            'dew', 'humidity', 'precip', 'precipprob', 'precipcover', 'preciptype', 'snow', 'snowdepth',
            'windgust', 'windspeed', 'winddir', 'pressure', 'cloudcover', 'visibility', 
            'solarradiation', 'solarenergy', 'uvindex', 'severerisk', 'sunrise', 'sunset', 
            'moonphase', 'conditions', 'description', 'icon'
        ]
        
        # Reindex and rename the DataFrame
        tmr_hourly_df = tmr_hourly_df.reindex(columns=columns_of_interest).rename(columns={"date": "Dates", "datetime": "Time"})
        
        return tmr_hourly_df
    else:
        print("Failed to retrieve data:", tmr_response.status_code)
        return None

# Example usage
tmr_weather_df = get_tmr_weather_df()
tmr_weather_df


Data retrieved successfully!
Weather data saved successfully!


,Dates,Time,temp,tempmax,tempmin,feelslike,feelslikemax,feelslikemin,dew,humidity,...,solarradiation,solarenergy,uvindex,severerisk,sunrise,sunset,moonphase,conditions,description,icon
0,2024-06-29,00:00:00,16.0,22.3,14.5,16.0,22.3,14.5,12.6,80.25,...,0.0,0.0,0.0,10.0,05:10:28,21:21:37,0.78,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day
1,2024-06-29,01:00:00,15.7,22.3,14.5,15.7,22.3,14.5,12.6,81.80,...,0.0,0.0,0.0,10.0,05:10:28,21:21:37,0.78,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day
2,2024-06-29,02:00:00,15.4,22.3,14.5,15.4,22.3,14.5,12.2,81.23,...,0.0,0.0,0.0,10.0,05:10:28,21:21:37,0.78,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day
3,2024-06-29,03:00:00,15.0,22.3,14.5,15.0,22.3,14.5,12.3,83.90,...,0.0,0.0,0.0,10.0,05:10:28,21:21:37,0.78,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day
4,2024-06-29,04:00:00,14.8,22.3,14.5,14.8,22.3,14.5,12.3,84.98,...,0.0,0.0,0.0,10.0,05:10:28,21:21:37,0.78,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day
5,2024-06-29,05:00:00,14.5,22.3,14.5,14.5,22.3,14.5,12.0,84.95,...,0.0,0.0,0.0,10.0,05:10:28,21:21:37,0.78,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day
6,2024-06-29,06:00:00,14.5,22.3,14.5,14.5,22.3,14.5,12.5,87.79,...,41.0,0.1,0.0,10.0,05:10:28,21:21:37,0.78,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day
7,2024-06-29,07:00:00,15.6,22.3,14.5,15.6,22.3,14.5,12.9,83.96,...,116.0,0.4,1.0,10.0,05:10:28,21:21:37,0.78,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day
8,2024-06-29,08:00:00,16.9,22.3,14.5,16.9,22.3,14.5,13.3,79.33,...,155.0,0.6,2.0,10.0,05:10:28,21:21:37,0.78,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day
9,2024-06-29,09:00:00,18.0,22.3,14.5,18.0,22.3,14.5,13.3,74.01,...,350.0,1.3,4.0,10.0,05:10:28,21:21:37,0.78,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day


In [46]:
# function to store yesterday's weather
yst_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/yesterday?unitGroup=metric&key=FLBEV2T2WUAPQGP2MP25GFASV&contentType=json"

yst_date = datetime.now() - timedelta(days=1)
yst_response = requests.get(yst_url)

# Check if the request was successful
if yst_response.status_code == 200:
    # Parse the JSON data
    weather_data = yst_response.json()
    print("Data retrieved successfully!")
    
    # File path setup for saving the data
    yst_file_path = f"{yst_date.strftime('%Y-%m-%d')}_weather_data.json"
    with open(yst_file_path, 'w') as f:
        json.dump(weather_data, f, indent=4)
    print("Weather data saved successfully!")

Data retrieved successfully!
Weather data saved successfully!


In [77]:
# Load the JSON data for the previous day
with open(yst_file_path, 'r') as file:
    json_data = json.load(file)

# Prepare new data from JSON to match the DataFrame structure
new_rows = []
for day in json_data['days']:
    for hour in day['hours']:
        hour.update({
            'Dates': day['datetime'],
            'Time': hour['datetime'],
            # Include other necessary columns here
            'tempmax': day['tempmax'],
            'tempmin': day['tempmin'],
            'feelslikemax': day['feelslikemax'],
            'feelslikemin': day['feelslikemin'],
            'sunrise': day['sunrise'],
            'sunset': day['sunset'],
            'moonphase': day['moonphase'],
            'conditions': day['conditions'],
            'description': day['description'],
            'icon': day['icon']
        })
        new_rows.append(hour)



# Convert to DataFrame, handle 'Dates' and 'Time' formatting
new_df = pd.DataFrame(new_rows)
new_df['Dates'] = pd.to_datetime(new_df['Dates'], errors='coerce')
new_df['Time'] = pd.to_datetime(new_df['Time'], format='%H:%M:%S').dt.time
new_df.columns
# Combine with existing data
# combined_df = pd.concat([weather_df, new_df], ignore_index=True)

KeyError: 'Dates'

In [76]:
new_df['datetime'] = str(new_df['Dates']) + str(new_df['Time'])
new_df

,datetime,datetimeEpoch,temp,feelslike,humidity,dew,precip,precipprob,snow,snowdepth,...,Dates,Time,tempmax,tempmin,feelslikemax,feelslikemin,sunrise,sunset,moonphase,description
0,0 2024-06-27\n1 2024-06-27\n2 2024-06...,1719471600,12.1,12.1,99.19,12.0,0.000,0.0,0.0,0.0,...,2024-06-27,00:00:00,14.0,12.0,14.0,12.0,05:09:23,21:21:57,0.71,Partly cloudy throughout the day with a chance...
1,0 2024-06-27\n1 2024-06-27\n2 2024-06...,1719475200,12.0,12.0,98.99,11.9,0.000,0.0,0.0,0.0,...,2024-06-27,01:00:00,14.0,12.0,14.0,12.0,05:09:23,21:21:57,0.71,Partly cloudy throughout the day with a chance...
2,0 2024-06-27\n1 2024-06-27\n2 2024-06...,1719478800,12.4,12.4,95.97,11.8,0.000,0.0,0.0,0.0,...,2024-06-27,02:00:00,14.0,12.0,14.0,12.0,05:09:23,21:21:57,0.71,Partly cloudy throughout the day with a chance...
3,0 2024-06-27\n1 2024-06-27\n2 2024-06...,1719482400,12.4,12.4,93.94,11.4,2.114,100.0,0.0,0.0,...,2024-06-27,03:00:00,14.0,12.0,14.0,12.0,05:09:23,21:21:57,0.71,Partly cloudy throughout the day with a chance...
4,0 2024-06-27\n1 2024-06-27\n2 2024-06...,1719486000,12.3,12.3,91.68,11.0,0.000,0.0,0.0,0.0,...,2024-06-27,04:00:00,14.0,12.0,14.0,12.0,05:09:23,21:21:57,0.71,Partly cloudy throughout the day with a chance...
5,0 2024-06-27\n1 2024-06-27\n2 2024-06...,1719489600,12.2,12.2,95.13,11.4,1.733,100.0,0.0,0.0,...,2024-06-27,05:00:00,14.0,12.0,14.0,12.0,05:09:23,21:21:57,0.71,Partly cloudy throughout the day with a chance...
6,0 2024-06-27\n1 2024-06-27\n2 2024-06...,1719493200,12.1,12.1,96.59,11.6,0.988,100.0,0.0,0.0,...,2024-06-27,06:00:00,14.0,12.0,14.0,12.0,05:09:23,21:21:57,0.71,Partly cloudy throughout the day with a chance...
7,0 2024-06-27\n1 2024-06-27\n2 2024-06...,1719496800,12.5,12.5,96.66,12.0,0.338,100.0,0.0,0.0,...,2024-06-27,07:00:00,14.0,12.0,14.0,12.0,05:09:23,21:21:57,0.71,Partly cloudy throughout the day with a chance...
8,0 2024-06-27\n1 2024-06-27\n2 2024-06...,1719500400,13.0,13.0,92.80,11.8,0.000,0.0,0.0,0.0,...,2024-06-27,08:00:00,14.0,12.0,14.0,12.0,05:09:23,21:21:57,0.71,Partly cloudy throughout the day with a chance...
9,0 2024-06-27\n1 2024-06-27\n2 2024-06...,1719504000,12.7,12.7,99.08,12.6,0.021,100.0,0.0,0.0,...,2024-06-27,09:00:00,14.0,12.0,14.0,12.0,05:09:23,21:21:57,0.71,Partly cloudy throughout the day with a chance...


In [73]:
new_df[['datetime', 'Dates', 'Time']]

,datetime,Dates,Time
0,00:00:00,2024-06-27,00:00:00
1,01:00:00,2024-06-27,01:00:00
2,02:00:00,2024-06-27,02:00:00
3,03:00:00,2024-06-27,03:00:00
4,04:00:00,2024-06-27,04:00:00
5,05:00:00,2024-06-27,05:00:00
6,06:00:00,2024-06-27,06:00:00
7,07:00:00,2024-06-27,07:00:00
8,08:00:00,2024-06-27,08:00:00
9,09:00:00,2024-06-27,09:00:00


In [66]:
a = pd.read_csv("weather_20240627.csv").drop(columns = 'Unnamed: 0')
a.columns

Index(['datetime', 'temp', 'feelslike', 'dew', 'humidity', 'precip',
       'precipprob', 'preciptype', 'snow', 'snowdepth', 'windgust',
       'windspeed', 'winddir', 'cloudcover', 'visibility', 'solarradiation',
       'solarenergy', 'uvindex', 'severerisk', 'conditions', 'icon',
       'stations'],
      dtype='object')

In [67]:
new_df_filtered = new_df[a.columns]
b = pd.concat([a, new_df_filtered], axis=0)
b

,datetime,temp,feelslike,dew,humidity,precip,precipprob,preciptype,snow,snowdepth,...,winddir,cloudcover,visibility,solarradiation,solarenergy,uvindex,severerisk,conditions,icon,stations
0,2024-06-06T00:00:00,11.6,11.6,7.1,73.78,0.0,0.0,NaN,0.0,0.0,...,11.0,0.0,32.0,0.0,0.0,0.0,10.0,Clear,clear-night,"71608099999,CWWA,71784099999,71892099999,71042..."
1,2024-06-06T01:00:00,11.4,11.4,7.3,76.18,0.0,0.0,NaN,0.0,0.0,...,35.0,30.0,32.0,0.0,0.0,0.0,10.0,Partially cloudy,partly-cloudy-night,"71608099999,CWWA,71784099999,71892099999,71042..."
2,2024-06-06T02:00:00,10.9,10.9,7.8,81.06,0.0,0.0,NaN,0.0,0.0,...,350.0,30.0,24.0,0.0,0.0,0.0,10.0,Partially cloudy,partly-cloudy-night,"71608099999,CWWA,71784099999,71892099999,71042..."
3,2024-06-06T03:00:00,10.9,10.9,7.7,80.54,0.0,0.0,NaN,0.0,0.0,...,16.0,50.0,32.0,0.0,0.0,0.0,10.0,Partially cloudy,partly-cloudy-night,"71608099999,CWWA,71784099999,71892099999,71042..."
4,2024-06-06T04:00:00,11.0,11.0,7.5,78.85,0.0,0.0,NaN,0.0,0.0,...,40.0,43.0,32.0,0.0,0.0,0.0,10.0,Partially cloudy,partly-cloudy-night,"71608099999,CWWA,71784099999,71892099999,71042..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19,19:00:00,13.7,13.7,12.1,89.67,0.0,0.0,None,0.0,0.0,...,22.0,74.0,16.2,65.0,0.2,1.0,10.0,"Rain, Partially cloudy",rain,"[CWWA, CWSK, CWEL, CWEZ, CWWK, CWMM, F1856]"
20,20:00:00,13.4,13.4,11.8,89.65,0.0,0.0,None,0.0,0.0,...,355.0,67.0,16.3,28.0,0.1,0.0,10.0,"Rain, Partially cloudy",rain,"[CWWA, CWSK, CWEL, CWEZ, CWWK, CWMM, F1856]"
21,21:00:00,13.4,13.4,12.1,91.82,0.0,0.0,None,0.0,0.0,...,39.0,66.0,16.3,3.0,0.0,0.0,10.0,"Rain, Partially cloudy",rain,"[CWWA, CWSK, CWEL, CWEZ, CWWK, CWMM, F1856]"
22,22:00:00,13.2,13.2,11.8,91.55,0.0,0.0,None,0.0,0.0,...,20.0,58.0,16.2,0.0,0.0,0.0,10.0,"Rain, Partially cloudy",rain,"[CWWA, CWSK, CWEL, CWEZ, CWWK, CWMM, F1856]"


In [68]:
def update_weather_data(previous_csv_path, json_data_path):
    # Load the existing CSV data if available
    if os.path.exists(previous_csv_path):
        weather_df = pd.read_csv(previous_csv_path)
        # Ensure 'Dates' is in the correct datetime format
        weather_df['Dates'] = pd.to_datetime(weather_df['Dates'], errors='coerce')
        # Drop any rows where 'Dates' could not be converted
        weather_df.dropna(subset=['Dates'], inplace=True)
    else:
        print("No previous data file found, starting new dataset.")
        weather_df = pd.DataFrame()

    # Load the JSON data for the previous day
    with open(json_data_path, 'r') as file:
        json_data = json.load(file)

    # Prepare new data from JSON to match the DataFrame structure
    new_rows = []
    for day in json_data['days']:
        for hour in day['hours']:
            hour.update({
                'Dates': day['datetime'],
                'Time': hour['datetime'],
                # Include other necessary columns here
                'tempmax': day['tempmax'],
                'tempmin': day['tempmin'],
                'feelslikemax': day['feelslikemax'],
                'feelslikemin': day['feelslikemin'],
                'sunrise': day['sunrise'],
                'sunset': day['sunset'],
                'moonphase': day['moonphase'],
                'conditions': day['conditions'],
                'description': day['description'],
                'icon': day['icon']
            })
            new_rows.append(hour)

    # Convert to DataFrame, handle 'Dates' and 'Time' formatting
    new_df = pd.DataFrame(new_rows)
    new_df['Dates'] = pd.to_datetime(new_df['Dates'], errors='coerce')
    new_df['Time'] = pd.to_datetime(new_df['Time'], format='%H:%M:%S').dt.time

    weather_df = weather_df.drop(columns = 'Unnamed: 0')
    new_df_filtered = new_df[weather_df.columns]
    combined_df = pd.concat([weather_df, new_df_filtered], axis=0)

    # Check for empty DataFrame
    if combined_df.empty:
        print("No data available to process.")
        return

    # Define new filename based on the date range
    start_date = combined_df['Dates'].min().strftime('%Y%m%d')
    end_date = combined_df['Dates'].max().strftime('%Y%m%d')
    new_filename = f"weather_{start_date}_to_{end_date}.csv"
    new_file_path = f'{new_filename}'

    # Save the combined data to a new file
    combined_df.to_csv(new_file_path, index=False)

    # Optionally delete the old CSV file if it's different from the new file
    if previous_csv_path != new_file_path and os.path.exists(previous_csv_path):
        os.remove(previous_csv_path)
        print(f"Deleted old data file: {previous_csv_path}")

    print(f"Data combined and saved successfully as {new_filename}")

# Example usage
yst_date = datetime.now() - timedelta(days=1)
yst_date_str = yst_date.strftime('%Y-%m-%d')
previous_csv = f'weather_{yst_date.strftime("%Y%m%d")}.csv'  # Update format as needed
json_path = f'{yst_date_str}_weather_data.json'
update_weather_data(previous_csv, json_path)


KeyError: 'Dates'

> 4.20~6.27 Weather Data

In [103]:
weather2024_0420_0626 = pd.read_csv("weather2024_0420_0626.csv").sort_values("datetime").drop(columns='Unnamed: 0')
a = pd.read_csv('vancouver 2024-06-27 to 2024-06-30.csv')
weather2024_0420_0630 = pd.concat([weather2024_0420_0626, a])
weather2024_0420_0630

,name,datetime,temp,feelslike,dew,humidity,precip,precipprob,preciptype,snow,...,sealevelpressure,cloudcover,visibility,solarradiation,solarenergy,uvindex,severerisk,conditions,icon,stations
1248,vancouver,2024-04-20T00:00:00,11.0,11.0,-0.6,44.45,0.0,0,NaN,0,...,1018.5,12.0,32.0,0,0.0,0,10,Clear,clear-night,"71608099999,CWWA,71784099999,71892099999,71042..."
1249,vancouver,2024-04-20T01:00:00,9.8,9.8,2.7,61.36,0.0,0,NaN,0,...,1018.6,0.0,32.0,0,0.0,0,10,Clear,clear-night,"71608099999,CWWA,71784099999,71892099999,71042..."
1250,vancouver,2024-04-20T02:00:00,9.8,9.3,2.4,59.86,0.0,0,NaN,0,...,1018.3,12.0,30.0,0,0.0,0,10,Clear,clear-night,"71608099999,CWWA,71784099999,71892099999,71042..."
1251,vancouver,2024-04-20T03:00:00,9.4,8.8,1.0,56.04,0.0,0,NaN,0,...,1017.8,0.0,32.0,0,0.0,0,10,Clear,clear-night,"71608099999,CWWA,71784099999,71892099999,71042..."
1252,vancouver,2024-04-20T04:00:00,9.0,9.0,2.7,64.70,0.0,0,NaN,0,...,1017.5,2.0,32.0,0,0.0,0,10,Clear,clear-night,"71608099999,CWWA,71784099999,71892099999,71042..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,vancouver,2024-06-30T19:00:00,18.8,18.8,13.4,70.90,0.0,0,NaN,0,...,1019.5,13.0,16.3,63,0.2,1,10,Clear,clear-day,"CWWA,CWWK,CWMM,F1856"
92,vancouver,2024-06-30T20:00:00,19.0,19.0,13.0,68.26,0.0,0,NaN,0,...,1019.2,12.0,16.3,55,0.2,1,10,Clear,clear-day,"CWWA,CWWK,CWMM,F1856"
93,vancouver,2024-06-30T21:00:00,18.8,18.8,13.4,70.55,0.0,0,NaN,0,...,1019.2,12.0,16.3,6,0.0,0,10,Clear,clear-day,"CWWA,CWWK,CWMM,F1856"
94,vancouver,2024-06-30T22:00:00,17.8,17.8,12.5,71.23,0.0,0,NaN,0,...,1019.5,14.0,16.3,0,0.0,0,10,Clear,clear-night,"CWWA,CWWK,CWMM,F1856"


In [104]:
weather2024_0420_0630['icon'].value_counts()

icon
partly-cloudy-day      675
rain                   354
partly-cloudy-night    302
clear-day              150
clear-night            149
cloudy                  98
Name: count, dtype: int64

In [97]:
weather2024_0420_0630['datetime'] = pd.to_datetime(weather2024_0420_0630['datetime'])
weather2024_0420_0630['date'] = weather2024_0420_0630['datetime'].dt.date
weather2024_0420_0630['time'] = weather2024_0420_0630['datetime'].dt.time
weather2024_0420_0630.columns

Index(['name', 'datetime', 'temp', 'feelslike', 'dew', 'humidity', 'precip',
       'precipprob', 'preciptype', 'snow', 'snowdepth', 'windgust',
       'windspeed', 'winddir', 'sealevelpressure', 'cloudcover', 'visibility',
       'solarradiation', 'solarenergy', 'uvindex', 'severerisk', 'conditions',
       'icon', 'stations', 'date', 'time'],
      dtype='object')

In [98]:
weather2024_0420_0630

,name,datetime,temp,feelslike,dew,humidity,precip,precipprob,preciptype,snow,...,visibility,solarradiation,solarenergy,uvindex,severerisk,conditions,icon,stations,date,time
1248,vancouver,2024-04-20 00:00:00,11.0,11.0,-0.6,44.45,0.0,0,NaN,0,...,32.0,0,0.0,0,10,Clear,clear-night,"71608099999,CWWA,71784099999,71892099999,71042...",2024-04-20,00:00:00
1249,vancouver,2024-04-20 01:00:00,9.8,9.8,2.7,61.36,0.0,0,NaN,0,...,32.0,0,0.0,0,10,Clear,clear-night,"71608099999,CWWA,71784099999,71892099999,71042...",2024-04-20,01:00:00
1250,vancouver,2024-04-20 02:00:00,9.8,9.3,2.4,59.86,0.0,0,NaN,0,...,30.0,0,0.0,0,10,Clear,clear-night,"71608099999,CWWA,71784099999,71892099999,71042...",2024-04-20,02:00:00
1251,vancouver,2024-04-20 03:00:00,9.4,8.8,1.0,56.04,0.0,0,NaN,0,...,32.0,0,0.0,0,10,Clear,clear-night,"71608099999,CWWA,71784099999,71892099999,71042...",2024-04-20,03:00:00
1252,vancouver,2024-04-20 04:00:00,9.0,9.0,2.7,64.70,0.0,0,NaN,0,...,32.0,0,0.0,0,10,Clear,clear-night,"71608099999,CWWA,71784099999,71892099999,71042...",2024-04-20,04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,vancouver,2024-06-30 19:00:00,18.8,18.8,13.4,70.90,0.0,0,NaN,0,...,16.3,63,0.2,1,10,Clear,clear-day,"CWWA,CWWK,CWMM,F1856",2024-06-30,19:00:00
92,vancouver,2024-06-30 20:00:00,19.0,19.0,13.0,68.26,0.0,0,NaN,0,...,16.3,55,0.2,1,10,Clear,clear-day,"CWWA,CWWK,CWMM,F1856",2024-06-30,20:00:00
93,vancouver,2024-06-30 21:00:00,18.8,18.8,13.4,70.55,0.0,0,NaN,0,...,16.3,6,0.0,0,10,Clear,clear-day,"CWWA,CWWK,CWMM,F1856",2024-06-30,21:00:00
94,vancouver,2024-06-30 22:00:00,17.8,17.8,12.5,71.23,0.0,0,NaN,0,...,16.3,0,0.0,0,10,Clear,clear-night,"CWWA,CWWK,CWMM,F1856",2024-06-30,22:00:00


In [102]:
weather2024_0420_0630['icon'].value_counts()

KeyError: 'icon'

In [99]:
weather2024_0420_0630 = weather2024_0420_0630[['date', 'time', 'temp', 'feelslike', 'dew', 'humidity', 'precip',
       'precipprob', 'preciptype', 'snow', 'snowdepth', 'windgust',
       'windspeed', 'winddir', 'cloudcover', 'solarradiation', 'solarenergy', 'uvindex', 'severerisk', 'conditions']]
weather2024_0420_0630 = weather2024_0420_0630.rename(columns={'date': 'Dates', 'time': 'Time'})
weather2024_0420_0630.to_csv('weather_20240420_to_20240630.csv')


## 2. Preprocess
### 2.1 Define sunset/sunrise --> "Day/Night" column !!! Change!!! According to the weather data

In [10]:
all_SR['Dates'] = pd.to_datetime(all_SR['Dates'])
all_SR['Time'] = pd.to_datetime(all_SR['Time'], format='%H:%M:%S').dt.time

# define average sunrise and sunset times for Vancouver by month
sun_times = {
    'Jun': ('05:06', '21:10'),
    'May': ('05:12', '20:30'),
    'Apr': ('05:51', '19:45'),
    'Mar': ('06:37', '17:57'),
    'Feb': ('06:55', '17:09'),
    'Jan': ('07:45', '16:25'),
    'Dec': ('07:46', '16:14'),
    'Nov': ('07:07', '16:17'),
    'Oct': ('07:12', '17:53'),
    'Sep': ('06:28', '18:53'),
    'Aug': ('05:44', '19:58'),
    'Jul': ('05:11', '20:54')
}

# classify each row
def classify_day_night(row, sun_times):
    month = row['Dates'].strftime('%b')  # month abbreviation
    sunrise, sunset = sun_times[month]
    sunrise = datetime.strptime(sunrise, '%H:%M').time()
    sunset = datetime.strptime(sunset, '%H:%M').time()
    
    if sunrise <= row['Time'] <= sunset:
        return 'Daytime'
    else:
        return 'Nighttime'

# apply the function
all_SR['Day/Night'] = all_SR.apply(lambda row: classify_day_night(row, sun_times), axis=1)
# 0 if nighttime
all_SR.loc[all_SR['Day/Night'] == 'Nighttime', 'SR'] = 0
all_SR

,Dates,Time,SR,Day/Night
616621,2024-04-20,13:07:58,82.57,Daytime
613949,2024-04-20,04:01:45,0.00,Nighttime
613950,2024-04-20,04:01:51,0.00,Nighttime
613951,2024-04-20,04:01:56,0.00,Nighttime
613952,2024-04-20,04:02:01,0.00,Nighttime
...,...,...,...,...
594642,2024-06-27,23:57:21,0.00,Nighttime
594643,2024-06-27,23:48:55,0.00,Nighttime
594644,2024-06-27,23:57:15,0.00,Nighttime
594646,2024-06-27,23:54:36,0.00,Nighttime
